# 06 — Menambah Fitur Latitude & Longitude

apakah model multi-station (data 7 stasiun digabung) yang dilengkapi koordinat lat/lon meningkatkan skill dibanding model per-stasiun?

disini 
- Gabungkan dataset semua stasiun sehat menjadi satu dataset latih, tambahkan kolom `lat`, `lon`.
- Bandingkan: (a) tanpa lat/lon, (b) dengan lat/lon.
- Karena ini multi-station, evaluasi dilakukan per-stasiun pada data test.

**anti-leakage spasial:** sequence LSTM dibuat **per-stasiun terlebih dahulu**, baru di concat. Ini agar satu window LSTM berisi N hari stasiun A + (lookback−N) hari stasiun B yang tidak bermakna sebagai deret waktu.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd

from src import config as C
from src.data_loader import load_all_stations
from src.evaluation import compute_metrics
from src.model import build_lstm, set_seed, train_model
from src.preprocessing import (
    chronological_split, fit_scaler_on_train, transform,
    impute_linear_then_fill, inverse_target, make_sequences,
)

In [2]:
BASE_FEATURES = C.WEATHER_COLS + [C.AOD_COL, C.TARGET]
FIXED_PARAMS = {'lstm_units': 32, 'dropout_rate': 0.1, 'optimizer': 'adam', 'learning_rate': 1e-3}
EPOCHS = 100
PATIENCE = 15

dfs = load_all_stations(reindex=True)

In [3]:
def prep_per_station(df, features):
    """Imputasi → split kronologis untuk satu stasiun."""
    df = impute_linear_then_fill(df, features).dropna(subset=features).reset_index(drop=True)
    return chronological_split(df)


def _sequences_per_station(parts, feat, scaler, target_idx):
    """Buat sequence per-stasiun lalu concat agar window TIDAK melintasi batas stasiun."""
    Xs, ys = [], []
    for part in parts:
        if len(part) <= C.LOOKBACK:
            continue  # sebuah split mungkin terlalu pendek untuk lookback
        arr = transform(part, feat, scaler)
        Xp, yp = make_sequences(arr, target_idx)
        Xs.append(Xp)
        ys.append(yp)
    return np.concatenate(Xs, axis=0), np.concatenate(ys, axis=0)


def build_combined(features, with_coords: bool):
    """Bangun train/val/test gabungan multi-stasiun (sequence per-stasiun, scaler fit di train)."""
    feat = list(features) + (['lat', 'lon'] if with_coords else [])
    target_idx = feat.index(C.TARGET)

    train_parts, val_parts, test_meta = [], [], {}
    for s in C.HEALTHY_STATIONS:
        df = dfs[s].copy()
        df['lat'] = C.STATIONS[s]['lat']
        df['lon'] = C.STATIONS[s]['lon']
        splits = prep_per_station(df, feat)
        train_parts.append(splits.train[feat])
        val_parts.append(splits.val[feat])
        test_meta[s] = splits.test[feat].copy()

    # Scaler fit pada gabungan TRAIN saja (no leakage dari val/test)
    train_concat = pd.concat(train_parts, ignore_index=True)
    scaler = fit_scaler_on_train(train_concat, feat)

    # Sequence per-stasiun → concat (TIDAK melintasi batas stasiun)
    X_tr, y_tr = _sequences_per_station(train_parts, feat, scaler, target_idx)
    X_va, y_va = _sequences_per_station(val_parts,   feat, scaler, target_idx)

    # Test tetap per-stasiun untuk evaluasi terpisah
    test_X, test_y = {}, {}
    for s, df_test in test_meta.items():
        if len(df_test) <= C.LOOKBACK:
            continue
        te = transform(df_test, feat, scaler)
        Xs, ys = make_sequences(te, target_idx)
        test_X[s] = Xs
        test_y[s] = ys

    return {
        'X_train': X_tr, 'y_train': y_tr,
        'X_val':   X_va, 'y_val':   y_va,
        'test_X':  test_X, 'test_y': test_y,
        'scaler':  scaler, 'feature_cols': feat, 'target_idx': target_idx,
    }

In [4]:
# Sanity check: jumlah window train harus = sum(per-stasiun (n_train_i - lookback))
data_check = build_combined(BASE_FEATURES, with_coords=False)
expected = sum(
    max(0, len(prep_per_station(dfs[s].assign(lat=C.STATIONS[s]['lat'], lon=C.STATIONS[s]['lon']), BASE_FEATURES).train) - C.LOOKBACK)
    for s in C.HEALTHY_STATIONS
)
print('expected train windows:', expected)
print('actual   train windows:', len(data_check['X_train']))
assert len(data_check['X_train']) == expected, 'sequence per-stasiun tidak konsisten!'

expected train windows: 5159
actual   train windows: 5159


In [5]:
def run_combined(with_coords: bool):
    data = build_combined(BASE_FEATURES, with_coords=with_coords)
    set_seed()
    n_features = data['X_train'].shape[2]
    model = build_lstm(input_shape=(C.LOOKBACK, n_features), **FIXED_PARAMS)
    train_model(model, data['X_train'], data['y_train'], data['X_val'], data['y_val'],
                epochs=EPOCHS, patience=PATIENCE, verbose=0)

    rows = []
    for s in C.HEALTHY_STATIONS:
        if s not in data['test_X']:
            continue
        y_pred = model.predict(data['test_X'][s], verbose=0).flatten()
        y_pred = inverse_target(y_pred, data['scaler'], data['target_idx'], n_features)
        y_true = inverse_target(data['test_y'][s], data['scaler'], data['target_idx'], n_features)
        rows.append({'station': s, **compute_metrics(y_true, y_pred)})
    return pd.DataFrame(rows), model

In [6]:
df_no, model_no = run_combined(with_coords=False)
df_yes, model_yes = run_combined(with_coords=True)

df_no['variant']  = 'no_coords'
df_yes['variant'] = 'with_coords'
df_combined = pd.concat([df_no, df_yes], ignore_index=True)
df_combined.to_csv(C.METRICS_DIR / '06_lat_long_comparison.csv', index=False)
df_combined

,station,R2,MSE,RMSE,MAE,variant
0,us_embassy_1,0.607006,174.749608,13.219289,10.322586,no_coords
1,us_embassy_2,0.609139,251.197981,15.849227,11.291440,no_coords
2,bundaran_hi,0.675974,292.151753,17.092447,13.275941,no_coords
3,kelapa_gading,0.747588,133.743422,11.564749,8.432586,no_coords
4,jagakarsa,0.478032,177.118030,13.308570,9.122421,no_coords
5,lubang_buaya,0.467567,322.715368,17.964280,14.028956,no_coords
6,kebun_jeruk,0.713364,272.852167,16.518237,11.808765,no_coords
7,us_embassy_1,0.644839,157.926510,12.566881,9.741255,with_coords
8,us_embassy_2,0.628592,238.695756,15.449782,11.118487,with_coords
9,bundaran_hi,0.757984,218.208811,14.771893,11.066290,with_coords


In [7]:
pivot = df_combined.pivot(index='station', columns='variant', values=['R2', 'RMSE', 'MAE'])
pivot

R2                   RMSE                    MAE  \
variant       no_coords with_coords  no_coords with_coords  no_coords   
station                                                                 
bundaran_hi    0.675974    0.757984  17.092447   14.771893  13.275941   
jagakarsa      0.478032    0.540932  13.308570   12.480963   9.122421   
kebun_jeruk    0.713364    0.707362  16.518237   16.690269  11.808765   
kelapa_gading  0.747588    0.718858  11.564749   12.205186   8.432586   
lubang_buaya   0.467567    0.450966  17.964280   18.242190  14.028956   
us_embassy_1   0.607006    0.644839  13.219289   12.566881  10.322586   
us_embassy_2   0.609139    0.628592  15.849227   15.449782  11.291440   

                           
variant       with_coords  
station                    
bundaran_hi     11.066290  
jagakarsa        8.312714  
kebun_jeruk     11.383356  
kelapa_gading    9.152890  
lubang_buaya    14.052364  
us_embassy_1     9.741255  
us_embassy_2    11.118487

In [8]:
model_no.save(C.MODEL_DIR / '06_combined_no_coords.keras')
model_yes.save(C.MODEL_DIR / '06_combined_with_coords.keras')